In [ ]:
import yaml


dataset_name = "ContentWise_LastActionSplit"


yaml_file = f"data/hyperparameter_grids/{dataset_name}.yaml"

with open(yaml_file, 'r') as f:
    hyperparam_grid = yaml.safe_load(f)

print(f"Loaded hyperparameter grid from {yaml_file}")
print(f"Models: {[k for k in hyperparam_grid.keys() if k != 'Shared']}")

In [ ]:
import itertools
import os

dataset_name = hyperparam_grid["Shared"]["--dataset_name"][0]

output_dir = f"data/hyperparameter_grids/{dataset_name}"
os.makedirs(output_dir, exist_ok=True)

def convert_bools(params, val):
    if val is True:
        return f"{params}"
    elif val is False:
        return f"--no-{params.lstrip('-')}"
    elif val is None:
        return f"{params}"
    else:
        return f"{params}={val}"

commands = {}
for block in hyperparam_grid:
    if block == "Shared":
        continue
    model = hyperparam_grid[block].get("--model_name", [block])[0]
    params_to_export = [
        key for key in list(hyperparam_grid["Shared"].keys()) + list(hyperparam_grid[block].keys())
        if key != "--model_name"
    ]
    values = [
        hyperparam_grid[block][param] if param in hyperparam_grid[block] else hyperparam_grid["Shared"][param]
        for param in params_to_export
    ]
    model_commands = commands.setdefault(model, [])
    for combo in itertools.product(*values):
        param_str = " ".join(convert_bools(param, val) for param, val in zip(params_to_export, combo))
        cmd = f'python ./src/pipelines/run_experiment.py --model_name={model} {param_str}'
        if cmd not in model_commands:
            model_commands.append(cmd)

for model, model_commands in commands.items():
    filename = os.path.join(output_dir, f"commands_{model}.txt")
    with open(filename, "w", newline="\n") as f:
        for cmd in model_commands:
            f.write(cmd + "\n")
    print(f"{filename} (first 3 lines):")
    for cmd in model_commands[:3]:
        print(cmd)
    print(f"Total commands for {model}: {len(model_commands)}\n")